# Imports


In [2]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, TextDataset, DataCollatorForLanguageModeling

# Check GPU availability


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# Define model name


In [4]:
model_name = "gpt2"

# Load the tokenizer and model


In [5]:
cache_dir = "./models_cache"

tokenizer = GPT2Tokenizer.from_pretrained(model_name, cache_dir=cache_dir)
model = GPT2LMHeadModel.from_pretrained(model_name, cache_dir=cache_dir)

# Add special tokens if needed


In [6]:
tokenizer.add_special_tokens({'pad_token': '[PAD]'})
model.resize_token_embeddings(len(tokenizer))
model.to(device)

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50258, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2SdpaAttention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50258, bias=False)
)

# Prepare the dataset


In [23]:
from datasets import load_dataset


# Load a text file as a dataset
train_dataset = load_dataset("text", data_files={"train": "./data/train.txt"})["train"]
test_dataset = load_dataset("text", data_files={"test": "./data/test.txt"})["test"]

print(f"Training dataset size: {len(train_dataset)}")
print(f"Testing dataset size: {len(test_dataset)}")


# Define a data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # GPT-2 is not trained with masked language modeling
)

Training dataset size: 23
Testing dataset size: 20


In [27]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask"])

Map:   0%|          | 0/23 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

# Define training arguments


In [24]:
training_args = TrainingArguments(
    output_dir="./gpt2_finetuned",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_dir="./logs",
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=500,
    logging_steps=10,
    load_best_model_at_end=True,
    fp16=True,  # Use mixed precision for faster training
    fp16_backend="auto",
    report_to="none"  # Disable reporting for simplicity
)

h:\Documents\Work\llm-zero-to-mastery\.venv\Lib\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


# Define a trainer


In [28]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)


C:\Users\Asus\AppData\Local\Temp\ipykernel_22744\1610494294.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


# Fine-tune the model


In [29]:
trainer.train()

  0%|          | 0/18 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

{'eval_loss': 3.851874589920044, 'eval_runtime': 0.2651, 'eval_samples_per_second': 75.454, 'eval_steps_per_second': 18.863, 'epoch': 1.0}
{'loss': 4.0137, 'grad_norm': 53.01753616333008, 'learning_rate': 6.000000000000001e-07, 'epoch': 1.67}


  0%|          | 0/5 [00:00<?, ?it/s]

{'eval_loss': 3.840672016143799, 'eval_runtime': 0.261, 'eval_samples_per_second': 76.625, 'eval_steps_per_second': 19.156, 'epoch': 2.0}


  0%|          | 0/5 [00:00<?, ?it/s]

{'eval_loss': 3.8120994567871094, 'eval_runtime': 0.2478, 'eval_samples_per_second': 80.7, 'eval_steps_per_second': 20.175, 'epoch': 3.0}


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


{'train_runtime': 11.0245, 'train_samples_per_second': 6.259, 'train_steps_per_second': 1.633, 'train_loss': 3.964896625942654, 'epoch': 3.0}


TrainOutput(global_step=18, training_loss=3.964896625942654, metrics={'train_runtime': 11.0245, 'train_samples_per_second': 6.259, 'train_steps_per_second': 1.633, 'total_flos': 18029150208000.0, 'train_loss': 3.964896625942654, 'epoch': 3.0})

# Save the model locally


In [30]:
trainer.save_model("./gpt2_finetuned")
tokenizer.save_pretrained("./gpt2_finetuned")

('./gpt2_finetuned\\tokenizer_config.json',
 './gpt2_finetuned\\special_tokens_map.json',
 './gpt2_finetuned\\vocab.json',
 './gpt2_finetuned\\merges.txt',
 './gpt2_finetuned\\added_tokens.json')

# Testing


In [36]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Load the saved model and tokenizer
model_path = "./gpt2_finetuned"
tokenizer = GPT2Tokenizer.from_pretrained(model_path)
model = GPT2LMHeadModel.from_pretrained(model_path)


In [37]:
prompt = "Once upon a time, there was a brave knight"

input_ids = tokenizer.encode(prompt, return_tensors="pt")


In [38]:
output = model.generate(
    input_ids,
    max_length=50,
    num_return_sequences=1,
    temperature=0.7,
    top_k=50,
    top_p=0.9,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id  # Explicitly set pad_token_id
)

# Decode the output
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print("Generated Text:")
print(generated_text)


Generated Text:
Once upon a time, there was a brave knight of the knights. He fought bravely, and fought bravely for the people of the Kingdom of Fëanor, and for the people of the Empire of Fëanor.




# Evaluate the Model


In [63]:
test_prompts = [
    "What is the capital of France?",
    "Can you tell me a joke?",
    "Explain machine learning in simple terms."
]

for prompt in test_prompts:
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    output = model.generate(
    input_ids,
    max_length=50,
    num_return_sequences=1,
    temperature=0.7,
    top_k=50,
    top_p=0.9,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id  # Explicitly set pad_token_id
)

    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"Prompt: {prompt}")
    print(f"Generated Response: {generated_text}\n")


Prompt: What is the capital of France?
Generated Response: What is the capital of France?

The capital of France is Marseille, the capital of France. Marseille is the capital of France. The capital of France is St. Petersburg, the capital of Russia. St. Petersburg is the capital

Prompt: Can you tell me a joke?
Generated Response: Can you tell me a joke? A joke about the "carpet bombing?" Or a joke about the "bombs" or "militarization?" Or a joke about the "bombs" or "militarization" of

Prompt: Explain machine learning in simple terms.
Generated Response: Explain machine learning in simple terms.

What's the big deal about machine learning?

Machine learning is a new type of machine learning. Machine learning is a way to understand complex data in terms of its structure and complexity. In this

